# ③ 第三階段：回測與績效驗證 (Backtesting & Performance Validation)

本 Notebook 專注於專案計畫「③ 第三階段：回測與績效驗證」的內容。主要任務包含：

1.  **載入模型與回測資料**:
    *   載入在第二階段訓練好的模型 (簡易模型或進階模型)。
    *   準備用於回測的資料集 (通常是測試集或一段獨立的、模型未見過的歷史資料)。
    *   確保回測資料經過與模型訓練時一致的預處理和特徵工程。
2.  **模型預測與訊號生成**:
    *   使用載入的模型對回測資料進行預測。
    *   根據模型的預測結果生成交易訊號 (例如：買進、賣出、觀望)。
3.  **3.1 簡單回測邏輯設計 (與實現)**:
    *   定義交易訊號如何觸發交易 (e.g., `prediction == 1` -> Buy, `prediction == -1` -> Sell)。
    *   定義進出場條件。
    *   設定初步的停損 (Stop-Loss) 與停利 (Take-Profit) 規則。
    *   考慮簡化的交易成本模型 (如固定點數的滑價和手續費)。
4.  **3.2 回測引擎初步實現**:
    *   編寫或使用現有的回測腳本/函數 (`backtest/run_backtest.py` 或類似功能)。
    *   在回測資料集上根據交易訊號和回測邏輯執行模擬交易。
5.  **3.3 績效指標計算與分析**:
    *   計算並分析各項關鍵績效指標 (KPIs)，例如：
        *   總損益 (PnL) 及損益曲線 (Equity Curve)。
        *   勝率 (Win Rate)。
        *   平均獲利/平均虧損 (Average Profit / Average Loss)。
        *   獲利因子 (Profit Factor)。
        *   最大回撤 (Max Drawdown)。
        *   夏普比率 (Sharpe Ratio)。
        *   交易次數。
    *   可視化回測結果，例如損益曲線圖、交易點位標記圖等。
6.  **(選做/進階) 3.4 Walk-Forward Validation 設計**:
    *   如果需要，設計並初步實現 Walk-Forward 驗證流程，以評估模型在不同時間段的穩定性。
7.  **(選做/進階) 3.5 參數化策略回測**:
    *   如果需要，建立機制以支援調整回測參數 (如停損點、進場閾值、交易成本等) 並比較不同參數組合下的回測結果。

---

In [ ]:
# 初始設定與導入必要函式庫
import pandas as pd
import numpy as np
import os
import sys
import joblib # 用於載入模型
import matplotlib.pyplot as plt
import plotly.graph_objects as go

# 將專案根目錄添加到 Python 搜尋路徑
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# 例如：從 utils 或 backtest 導入需要的模組
# # 載入資料相關
# from utils.data_loader import load_and_inspect_data # 或其他載入函數
# from utils.data_cleaner import clean_data, process_time_series
# from features.feature_engineering import add_all_features # 假設有此函數統一處理特徵

# # 回測邏輯與績效計算相關
# from backtest.core_logic import run_backtest # 假設這是您的回測函數
# from utils.performance_metrics import calculate_performance, plot_pnl_curve # 假設這是您的績效計算與繪圖函數

## 1. 載入模型與準備回測資料
載入先前訓練好的模型 (簡易或進階)，並準備用於回測的資料集。
**重要：** 回測資料必須經過與模型訓練時完全相同的預處理和特徵工程流程。

In [ ]:
# 選擇要回測的模型
# model_to_backtest_path = '../trained_models/simple_logistic_model.joblib' # 快速原型模型
model_to_backtest_path = '../trained_models/advanced_xgb_pipeline.joblib' # 進階模型 Pipeline

try:
    model = joblib.load(model_to_backtest_path)
    print(f"模型 {model_to_backtest_path} 載入成功。")
except FileNotFoundError:
    print(f"錯誤：找不到模型檔案 {model_to_backtest_path}。請先執行相應的模型訓練 Notebook。")
    model = None

# 準備回測資料 (例如，使用一天或一段模型未見過的資料)
# 假設我們使用 ticks_2025-0528.parquet 作為回測資料
backtest_data_file = '../data/ticks_2025-0528.parquet' # 或其他回測資料檔案
df_backtest_raw = pd.read_parquet(backtest_data_file)

# --- 對回測資料進行預處理與特徵工程 --- 
# 這裡的步驟需要與模型訓練時的特徵工程步驟完全一致
# 以下為示例流程，請根據您的實際 feature_engineering 和 data_cleaner 調整

# df_backtest = clean_data(df_backtest_raw.copy()) # 清洗
# df_backtest = process_time_series(df_backtest)    # 時間序列處理

# # 添加特徵 (假設有一個統一的函數 add_all_features)
# # 如果沒有，需要逐步調用 calculate_sma, calculate_ema, calculate_rsi 等所有特徵計算函數
# try:
#     from features.feature_engineering import add_all_features # 假設您已將特徵計算整合
#     df_backtest = add_all_features(df_backtest) 
# except ImportError:
#     print("警告: add_all_features 函數未找到，請手動添加所有必要特徵或檢查 features.feature_engineering 模組。")
#     # 手動添加特徵的範例 (需要與訓練時一致):
#     # df_backtest['SMA_20'] = calculate_sma(df_backtest['close'], window=20)
#     # ... 其他特徵 ...

# # 移除因計算指標 (如移動平均初期) 或標註 (如 shift) 產生的 NaN 值
# # 注意：如果模型 Pipeline 中有 Imputer，這裡的 dropna 策略可能需要調整
# df_backtest.dropna(inplace=True) 

# if df_backtest.empty:
#     print("錯誤：回測資料在預處理後為空，請檢查資料或預處理步驟。")
# else:
#     print(f"\n回測資料準備完成，共 {len(df_backtest)} 筆。")
#     print(df_backtest.head())

## 2. 模型預測與訊號生成
使用載入的模型對處理後的回測資料進行預測，並根據預測結果生成交易訊號。

In [ ]:
# if model and not df_backtest.empty:
#     # 提取模型需要的特徵欄位
#     # 如果模型是 Pipeline，它內部知道需要哪些原始特徵
#     # 如果模型是單獨的 XGBoost，需要確保 X_backtest 的欄位與訓練時一致
#     if hasattr(model, 'named_steps'): # 檢查是否為 Pipeline
#         # Pipeline 會自動處理其內部步驟所需的特徵
#         # 但我們仍需提供與訓練時結構相似的輸入
#         # 假設 Pipeline 的第一個 transformer (如 imputer 或 scaler) 可以從 df_backtest 中提取所需欄位
#         # 或者，如果 Pipeline 的 xgb 步驟有 feature_names_in_ 屬性
#         try:
#             if hasattr(model.named_steps.get('xgb'), 'feature_names_in_'):
#                  features_for_prediction = model.named_steps['xgb'].feature_names_in_
#             elif hasattr(model.named_steps.get('classifier'), 'feature_names_in_'): # for simple_logistic_model
#                  features_for_prediction = model.named_steps['classifier'].feature_names_in_
#             else: # Fallback: 嘗試從 X_train (如果之前有保存或可以重新生成) 推斷
#                  print("警告: 無法直接從 Pipeline 的模型步驟獲取 feature_names_in_，請確保 X_backtest 欄位正確。")
#                  # features_for_prediction = X_train.columns.tolist() # 需要 X_train 可用
#                  features_for_prediction = [f for f in df_backtest.columns if f not in ['timestamp', 'label', 'open', 'high', 'low', 'volume', 'amount', 'ticks', 'future_price', 'price_diff'] and not f.startswith('label_') and not f.startswith('target_')] # 粗略估計

#         except AttributeError:
#             print("警告: Pipeline 結構不符合預期，無法自動獲取特徵名稱。請手動指定 X_backtest 的欄位。")
#             features_for_prediction = [f for f in df_backtest.columns if f not in ['timestamp', 'label', 'open', 'high', 'low', 'volume', 'amount', 'ticks', 'future_price', 'price_diff'] and not f.startswith('label_') and not f.startswith('target_')] # 粗略估計
#     else: # 單獨模型
#         features_for_prediction = model.feature_names_in_
        
#     X_backtest_for_model = df_backtest[features_for_prediction]
#     predictions_raw = model.predict(X_backtest_for_model)
    
#     # 將模型的預測 (例如 0, 1, 2 for XGBoost) 轉換回交易訊號 (-1: Sell, 0: Neutral, 1: Buy)
#     # 這需要根據模型訓練時的標籤映射進行調整
#     # 假設 XGBoost 的 0 -> -1 (Sell), 1 -> 0 (Neutral), 2 -> 1 (Buy)
#     # 假設 Logistic Regression 的 -1, 0, 1 直接是訊號
#     if 'xgb' in str(type(model.named_steps.get('xgb', model))).lower() or 'xgb' in str(type(model)).lower(): # 檢查是否為XGBoost相關模型
#         signal_map = {0: -1, 1: 0, 2: 1} 
#     else: # 假設其他模型 (如 LogisticRegression) 的預測直接是 -1, 0, 1
#         signal_map = {-1: -1, 0: 0, 1: 1} 
        
#     df_backtest['prediction_signal'] = pd.Series(predictions_raw, index=X_backtest_for_model.index).map(signal_map)
    
#     print("\n模型預測完成，交易訊號已加入 DataFrame:")
#     print(df_backtest[['close', 'prediction_signal']].head())
#     print("訊號分佈:")
#     print(df_backtest['prediction_signal'].value_counts())

## 3. 執行回測
使用 `backtest/core_logic.py` 中的回測函數 (或自行實現的邏輯) 進行模擬交易。

In [ ]:
# trades_df = pd.DataFrame() # 初始化空的交易紀錄 DataFrame
# final_balance = 0.0

# if 'prediction_signal' in df_backtest.columns and not df_backtest.empty:
#     try:
#         from backtest.core_logic import run_backtest 
#         trades_df, final_balance = run_backtest(
#             df_backtest.copy(), # 傳遞 DataFrame 的副本
#             entry_signal_col='prediction_signal', 
#             price_col='close', # 使用收盤價進行交易
#             timestamp_col='timestamp', # 確保您的 DataFrame 有 'timestamp' 欄位作為索引或普通欄位
#             stop_profit_pips=10,    # 示例：停利10點 (pips)
#             stop_loss_pips=5,      # 示例：停損5點 (pips)
#             cost_per_trade_pips=0.5 # 示例：每筆交易成本0.5點 (滑價+手續費)
#         )
#         print(f"\n回測完成。共執行 {len(trades_df)} 筆交易。")
#         if not trades_df.empty:
#             print(f"最終帳戶餘額 (點數): {final_balance:.2f}")
#             print("交易紀錄範例:")
#             print(trades_df.head())
#         else:
#             print("回測未產生任何交易。")
#     except ImportError:
#         print("錯誤: backtest.core_logic.run_backtest 未找到。請確保該模組與函數存在。")
#     except Exception as e:
#         print(f"回測過程中發生錯誤: {e}")
# else:
#     print("\n無法執行回測：'prediction_signal' 欄位不存在或 df_backtest 為空。")

## 4. 計算與展示績效指標
使用 `utils/performance_metrics.py` 中的函數 (或自行實現的邏輯) 計算並展示回測績效。

In [ ]:
# if not trades_df.empty:
#     try:
#         from utils.performance_metrics import calculate_performance, plot_pnl_curve
#         performance_summary = calculate_performance(trades_df)
#         print("\n績效摘要:")
#         for metric, value in performance_summary.items():
#             # 格式化輸出，對於比率型指標顯示百分比
#             if metric in ['Win Rate', 'Max Drawdown']:
#                 print(f"  {metric}: {value:.2%}") 
#             elif isinstance(value, float):
#                 print(f"  {metric}: {value:.4f}")
#             else:
#                 print(f"  {metric}: {value}")
    
#         # 繪製 PnL 曲線
#         plot_pnl_curve(trades_df)

#     except ImportError:
#         print("錯誤: utils.performance_metrics 中的函數未找到。請確保該模組與函數存在。")
#     except Exception as e:
#         print(f"計算或繪製績效時發生錯誤: {e}")
# elif 'prediction_signal' in df_backtest.columns: # 如果有訊號但沒有交易
#      print("\n沒有交易產生，無法計算績效。請檢查您的交易邏輯、訊號或市場條件。")

## 5. (選做) Walk-Forward Validation 或參數化回測
如果需要，可以在此處擴展進行更複雜的回測分析。